# Data exploration and preprocessing checks
This notebook validates the gait data before model training. Each section explains what is being inspected and what that check achieves.

In [1]:
# pandas handles tabular gait data, NumPy supports numerical checks, and
# Matplotlib is available for visual inspection when plots are needed.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# List every gait class that should be included in the analysis.
gaits = ['normal', 'antalgic', 'stiff_legged', 'lurch', 'steppage', 'trendelenburg']
# Load one Parquet file per class and combine them into one DataFrame.
# This creates a single, consistently indexed dataset for later checks.
df = pd.concat([pd.read_parquet(f'../data/parquet/{g}.parquet') for g in gaits], ignore_index=True)
# Confirm the total number of rows and columns loaded.
df.shape

(340895, 83)

In [3]:
# Display representative metadata and coordinate columns from the first rows.
# This confirms that the files have the expected structure and values.
df.iloc[:, :10].head()

,subject_id,gait_label,walk_id,sensor_id,frame_index,source_file,n_frames,time,joint0_x,joint0_y
0,1,normal,1,1,0,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.040,-0.063138,-0.217130
1,1,normal,1,1,1,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.075,-0.041459,-0.186096
2,1,normal,1,1,2,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.107,-0.041850,-0.154736
3,1,normal,1,1,3,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.141,-0.040571,-0.123253
4,1,normal,1,1,4,human1_normal1/human1_normal1_SkeletonData1.csv,77,12.175,-0.045611,-0.095217


The first eight columns contain identifiers, labels, frame information, and time. The remaining 75 columns represent 25 joints with x, y, and z coordinates. This separation tells us which columns can be used as metadata and which contain movement signals.

In [ ]:
# Count columns by data type to detect unexpected types before modeling.

df.dtypes.value_counts()

float32    75
int64       4
str         3
float64     1
Name: count, dtype: int64

### Class balance


In [5]:
# Count distinct sensor files for every gait label.
# This estimates how many independent recordings are available per class.
df.groupby('gait_label')['source_file'].nunique()

gait_label
antalgic         472
lurch            460
normal           465
steppage         467
stiff_legged     470
trendelenburg    460
Name: source_file, dtype: int64

In [6]:
# Count distinct subjects in each class.
# This checks that class coverage is not coming from only a few people.
df.groupby('gait_label')['subject_id'].nunique()

gait_label
antalgic         10
lurch            10
normal           10
steppage         10
stiff_legged     10
trendelenburg    10
Name: subject_id, dtype: int64

Each walk has several sensor files, so one later training sample will represent one sensor file. Keeping this unit explicit helps avoid accidentally mixing frames from different recordings.

In [7]:
# Count all frames contributed by each sensor.
# This reveals whether some sensors contribute much more data than others.
df['sensor_id'].value_counts()

sensor_id
3    65410
5    60415
4    57668
6    57605
2    50438
1    49359
Name: count, dtype: int64

In [8]:
# Summarize the number of frames in each recording.
# The distribution shows recording-length variation that may need to be
# handled when sequences are converted into fixed-size model features.
df.groupby('source_file')['n_frames'].first().describe()

count    2794.000000
mean      122.009664
std        39.547280
min        52.000000
25%        90.000000
50%       117.000000
75%       145.000000
max       305.000000
Name: n_frames, dtype: float64

### Bad values
The build step warned about huge numbers and some NaNs, so these checks identify invalid rows before feature extraction. The counts show how much data needs cleaning or exclusion.

In [9]:
# Select only joint-coordinate columns, excluding metadata and labels.
coords = df.filter(regex='^joint')
# Count missing coordinate values across the entire dataset.
# This quantifies incomplete pose measurements.
coords.isna().sum().sum()

np.int64(29)

In [10]:
# Mark a frame as invalid if any coordinate is missing or has an
# implausibly large magnitude.
bad = coords.isna().any(axis=1) | (coords.abs() > 10).any(axis=1)
# Count invalid frames so preprocessing can remove or repair them later.
bad.sum()

np.int64(97)

In [11]:
# Count recordings containing at least one invalid frame.
# This shows whether the problem is isolated to a few files or widespread.
df[bad]['source_file'].nunique()

65

### A quick look at the signal

Comparing a coordinate summary across labels provides a first sanity check that the pose signals differ between gait types.

In [12]:
# Exclude invalid frames, then calculate the mean vertical position of one
# joint for each class. This is only a sanity check, not a final feature.
df[~bad].groupby('gait_label')['joint14_y'].mean()

gait_label
antalgic         0.641914
lurch            0.624802
normal           0.626882
steppage         0.588701
stiff_legged     0.594803
trendelenburg    0.577188
Name: joint14_y, dtype: float32